No-attention control: BiLSTM or BiGRU encoder over passage+question+answer, pool and concatenate, feed to an MLP classifier. Cheap, trains fast, and it doubles as your "attention vs no attention" ablation, so it's not wasted effort. - download - RAY 


Fine-tuned pretrained transformer: BERT-base, RoBERTa-base, or DistilBERT via HuggingFace, fine-tuned on MultiRC as sequence classification. - download of hugging face

In [1]:
import json

TRAIN_PATH = "./MultiRC/train_456-fixedIds.json"
TEST_PATH = "./MultiRC/test_83-fixedIds.json"

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_json = json.load(f)["data"]

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_json = json.load(f)["data"]

In [2]:
import pandas as pd
import re

def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()

def clean_paragraph(paragraph):
    sentences = [clean_text(re.sub(r"<b>Sent \d+: </b>", "", sentence)) for sentence in paragraph.split("<br>")]
    return " ".join([sentence for sentence in sentences if sentence != ""])

def build_df(data_dict):
    data_rows = []
    
    for passage in data_dict:
        para_id = passage["id"]
        paragraph_obj = passage["paragraph"]
        paragraph_text = clean_paragraph(paragraph_obj["text"])
        questions = paragraph_obj["questions"]
        for i, question in enumerate(questions):
            question_text = clean_text(question["question"])
            answers = question["answers"]
            for j, answer in enumerate(answers):
                answer_text = clean_text(answer["text"])
                label = int(answer["isAnswer"])
                data_row = {"para_id": para_id, "q_id": f"{para_id}_{i}", "a_id": j, "paragraph": paragraph_text, "question": question_text, "answer": answer_text, "qa": f"{question_text} [SEP] {answer_text}", "label": label}
                data_rows.append(data_row)

    df = pd.DataFrame(data_rows)    
    return df

train_df = build_df(train_json)
test_df = build_df(test_json)

In [3]:
from sklearn.model_selection import train_test_split

SEED = 42

train_para_ids, val_para_ids = train_test_split(train_df["para_id"].unique(), test_size=0.1, random_state=SEED)
val_df = train_df[train_df["para_id"].isin(val_para_ids)]
train_df = train_df[train_df["para_id"].isin(train_para_ids)]

In [4]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def apply_tokenizer(qa_list, paragraph_list):
    return tokenizer(qa_list, paragraph_list, truncation="only_second", max_length=512)

train_inputs = apply_tokenizer(train_df["qa"].to_list(), train_df["paragraph"].to_list())
val_inputs = apply_tokenizer(val_df["qa"].to_list(), val_df["paragraph"].to_list())
test_inputs = apply_tokenizer(test_df["qa"].to_list(), test_df["paragraph"].to_list())

In [5]:
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, inputs, labels):
        self.data = inputs
        self.labels = labels.to_list()

    def __len__(self):
        return len(self.data["input_ids"])

    def __getitem__(self, i):
        return {"input_ids": self.data["input_ids"][i], "attention_mask": self.data["attention_mask"][i], "labels": self.labels[i]}

train_dataset = CustomDataset(train_inputs, train_df["label"])
val_dataset = CustomDataset(val_inputs, val_df["label"])
test_dataset = CustomDataset(test_inputs, test_df["label"])

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from transformers import DistilBertForSequenceClassification

NUM_EPOCHS = 5
LEARNING_RATE = 2e-5

def set_seed(val):
    random.seed(val)
    np.random.seed(val)
    torch.manual_seed(val)
    torch.cuda.manual_seed_all(val)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased").to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader
from copy import deepcopy

collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True, return_tensors="pt")
train_loader = DataLoader(dataset=train_dataset, batch_size=16, collate_fn=collator, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=16, collate_fn=collator, shuffle=False)

max_val_acc = 0
best_model = None

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_correct = 0
    train_total = 0
    total_loss = 0
    
    for batch in train_loader:
        for key in batch.keys():
            batch[key] = batch[key].to(device)
        targets = batch["labels"]
        optimizer.zero_grad()
        outputs = model(**batch)
        logits = outputs.logits
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        preds = logits.argmax(dim=1)
        train_correct += (preds == targets).sum().item()
        train_total += targets.size(0)
        total_loss += loss.item()

    train_acc = train_correct / train_total
    mean_loss = total_loss / len(train_loader)

    model.eval()
    val_correct = 0
    val_total = 0

    for batch in val_loader:
        for key in batch.keys():
            batch[key] = batch[key].to(device)
        targets = batch["labels"]
        outputs = model(**batch)
        logits = outputs.logits
        preds = logits.argmax(dim=1)
        val_correct += (preds == targets).sum().item()
        val_total += targets.size(0)

    val_acc = val_correct / val_total
    if val_acc > max_val_acc:
        max_val_acc = val_acc
        best_model = deepcopy(model.state_dict())

    print(f"Epoch: {epoch}, loss: {round(mean_loss, 4)}, train_acc: {round(train_acc, 4)}, val_acc: {round(val_acc, 4)}")

print("Finished Training")

Epoch: 1, loss: 0.6646, train_acc: 0.5872, val_acc: 0.647
Epoch: 2, loss: 0.563, train_acc: 0.7025, val_acc: 0.6746
Epoch: 3, loss: 0.4647, train_acc: 0.7728, val_acc: 0.6907
Epoch: 4, loss: 0.3751, train_acc: 0.8267, val_acc: 0.6918
Epoch: 5, loss: 0.2898, train_acc: 0.8671, val_acc: 0.6754
Finished Training


In [9]:
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True, return_tensors="pt")
test_loader = DataLoader(dataset=test_dataset, batch_size=16, collate_fn=collator)

model.load_state_dict(best_model)
model.eval()

preds = []
with torch.no_grad():
    for batch in test_loader:
        for key in batch.keys():
            batch[key] = batch[key].to(device)
        outputs = model(**batch)
        logits = outputs.logits
        preds.extend(logits.argmax(dim=1).tolist())

print("Test Performance:")
targets = test_dataset.labels
print(classification_report(targets, preds, digits=4))

Test Performance:
              precision    recall  f1-score   support

           0     0.7280    0.6758    0.7010      2773
           1     0.6047    0.6627    0.6323      2075

    accuracy                         0.6702      4848
   macro avg     0.6664    0.6692    0.6666      4848
weighted avg     0.6752    0.6702    0.6716      4848



Open-source LLM, zero/few-shot: Flan-T5, Mistral-7B, or a quantized Llama, prompted rather than trained. These are open-weight so they're allowed; the ban is only on closed APIs like GPT/Claude/Gemini. Worth flagging: a 7B model on free Colab needs 4-bit quantization or you'll run out of memory. Flan-T5-large is the safer bet if compute's tight. - api 